In [1]:
from pathlib import Path
from sklearn.model_selection import train_test_split
from PIL import Image
import shutil
import random
import pandas as pd
import tensorflow as tf
import keras
import os, time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
from keras import layers, models

In [3]:

SOURCE_DIRS = [
    Path("archive\seg_test\seg_test"),
    Path("archive\seg_train\seg_train")
]

OUTPUT_DIR = Path("archive/intel_64_split")
IMAGE_SIZE = (64, 64)
SEED = 42

# Collect images from class folders
images = []
extensions = {".jpg", ".jpeg", ".png", ".bmp"}

for source in SOURCE_DIRS:
    if not source.exists():
        print("Check missing path:", source)
        continue

    for class_dir in source.iterdir():
        if class_dir.is_dir():
            for path in class_dir.rglob("*"):
                if path.suffix.lower() in extensions:
                    images.append((path, class_dir.name))

if not images:
    raise ValueError(
        "No images found. Check SOURCE_DIRS and folder structure."
    )

# Split into 70% training and 30% temporary
paths = [str(path) for path, label in images]
labels = [label for path, label in images]

train_paths, temp_paths, train_y, temp_y = train_test_split(
    paths, labels,
    test_size=0.30,
    random_state=SEED,
    stratify=labels
)

# Split the remaining 30% equally into validation and test
val_paths, test_paths, val_y, test_y = train_test_split(
    temp_paths, temp_y,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_y
)

splits = {
    "train": (train_paths, train_y),
    "val": (val_paths, val_y),
    "test": (test_paths, test_y)
}

# Resize and save RGB images
for split, (split_paths, split_labels) in splits.items():
    for i, (path, label) in enumerate(zip(split_paths, split_labels)):
        destination = OUTPUT_DIR / split / label / f"{i:06d}.jpg"
        destination.parent.mkdir(parents=True, exist_ok=True)

        try:
            with Image.open(path) as img:
                img.convert("RGB").resize(
                    IMAGE_SIZE, Image.Resampling.LANCZOS
                ).save(destination, quality=95)
        except Exception as e:
            print("Skipped:", path, e)

print("Dataset saved to:", OUTPUT_DIR)
print("Training images:", len(train_paths))
print("Validation images:", len(val_paths))
print("Testing images:", len(test_paths))

<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:3: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:3: SyntaxWarning: invalid escape sequence '\s'
C:\Users\Praveen\AppData\Local\Temp\ipykernel_12336\1960076233.py:2: SyntaxWarning: invalid escape sequence '\s'
  Path("archive\seg_test\seg_test"),
C:\Users\Praveen\AppData\Local\Temp\ipykernel_12336\1960076233.py:3: SyntaxWarning: invalid escape sequence '\s'
  Path("archive\seg_train\seg_train")


KeyboardInterrupt: 

Found 11923 files belonging to 6 classes.
Found 2555 files belonging to 6 classes.
Found 2556 files belonging to 6 classes.
['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']


In [8]:
NUM_CLASSES= 6
INPUT_SHAPE=(64,64,3)

In [3]:


#Model A: Standard CNN
def build_model_a():
    model = models.Sequential([
        layers.Input(shape=INPUT_SHAPE),

        layers.Conv2D(32, (3,3,), padding="same",
                      activation="relu"),

        layers.MaxPooling2D((2,2)),

           layers.Conv2D(64, (3, 3), padding="same",
                      activation="relu"),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), padding="same",
                      activation="relu"),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(NUM_CLASSES, activation="softmax")

    ], name="Model_A_Standard_CNN")

    return model

In [8]:
# Model B: Lightweight CNN
def build_model_b():
    model = models.Sequential([
        layers.Input(shape=INPUT_SHAPE),

        layers.SeparableConv2D(
            32, (3, 3), padding="same",
            activation="relu"
        ),
        layers.MaxPooling2D((2, 2)),

        layers.SeparableConv2D(
            64, (3, 3), padding="same",
            activation="relu"
        ),
        layers.MaxPooling2D((2, 2)),

        layers.SeparableConv2D(
            128, (3, 3), padding="same",
            activation="relu"
        ),
        layers.MaxPooling2D((2, 2)),

        layers.GlobalAveragePooling2D(),
        layers.Dense(64, activation="relu"),
        layers.Dense(NUM_CLASSES, activation="softmax")
    ], name="Model_B_Lightweight_CNN")

    return model

In [11]:
model_a = build_model_a()
model_b = build_model_b()

model_a.summary()
model_b.summary()

print("Model A parameters:", model_a.count_params())
print("Model B parameters:", model_b.count_params())

assert model_b.count_params() <= 100_000

Model: "Model_A_Standard_CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 64, 64, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 32, 32, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 16, 16, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │     1,048,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,142,726 (4.36 MB)

 Trainable params: 1,142,726 (4.36 MB)

 Non-trainable params: 0 (0.00 B)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ separable_conv2d_3              │ (None, 64, 64, 32)     │           155 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d_4              │ (None, 32, 32, 64)     │         2,400 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d_5              │ (None, 16, 16, 128)    │         8,896 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,097 (78.50 KB)

 Trainable params: 20,097 (78.50 KB)

 Non-trainable params: 0 (0.00 B)

Model A parameters: 1142726
Model B parameters: 20097
